---

## Problem 4: Steady-State Energy Balance on a Reactor Network

> **Suggested time: ~25 minutes**

Three well-mixed tanks are connected in series. Each tank exchanges heat with its neighbors and with an external heating jacket. At steady state, the temperature of each tank ($T_1$, $T_2$, $T_3$ in °C) is governed by an energy balance.

```
  Jacket (T_J = 120°C)
        │  UA_J = 3
        ▼
   ┌─────────┐   UA_12 = 2   ┌─────────┐   UA_23 = 1   ┌─────────┐
   │  Tank 1 ├───────────────►  Tank 2 ├───────────────►  Tank 3 │
   │  mCp=10 │               │  mCp=8  │               │  mCp=6  │
   └─────────┘               └─────────┘               └─────────┘
    T1,in=25°C                T2,in=25°C                T3,in=25°C
```

**Steady-state energy balance on each tank** (heat in = heat out via flow + exchange):

$$\dot{m}C_p(T_{i,\text{in}} - T_i) + \sum_j UA_{ij}(T_j - T_i) + UA_J(T_J - T_i)\delta_{i,1} = 0$$

where $\delta_{i,1} = 1$ only for Tank 1 (the jacket only heats Tank 1).

Expanding each tank's balance and rearranging to the form $(\ldots)T_i + (\ldots)T_j = \text{RHS}$:

**Tank 1** (exchanges with jacket and Tank 2):
$$-(10 + 2 + 3)T_1 + 2T_2 = -10(25) - 3(120) \quad\Longrightarrow\quad -15T_1 + 2T_2 = -610$$

**Tank 2** (exchanges with Tank 1 and Tank 3):
$$2T_1 - (8 + 2 + 1)T_2 + T_3 = -8(25) \quad\Longrightarrow\quad 2T_1 - 11T_2 + T_3 = -200$$

**Tank 3** (exchanges only with Tank 2):
$$T_2 - (6 + 1)T_3 = -6(25) \quad\Longrightarrow\quad T_2 - 7T_3 = -150$$

**Tasks:**

**(a)** Assemble the coefficient matrix $A$ and right-hand-side vector $\mathbf{b}$ for the system $A\mathbf{T} = \mathbf{b}$ where $\mathbf{T} = [T_1, T_2, T_3]^T$.

**(b)** Verify the matrix is non-singular, then solve for $T_1$, $T_2$, $T_3$.

**(c)** Verify the solution with `A @ T_sol ≈ b`.

**(d)** Physical interpretation — answer in comments:
1. Why is $T_1$ higher than $T_2$ and $T_3$? 
2. Which tank has the lowest temperature, and why does heat flow in that direction?
3. What would happen to all three temperatures if you increased the jacket temperature $T_J$ from 120°C to 200°C? (You don't need to re-solve — just reason physically, then optionally verify by changing the code.)

In [ ]:
import numpy as np

# (a) Assemble A and b
A = np.array([[-15,   2,   0],   # Tank 1 balance
              [  2, -11,   1],   # Tank 2 balance
              [  0,   1,  -7]])  # Tank 3 balance

b = np.array([-610.0, -200.0, -150.0])

print("A =\n", A)
print("b =", b)

# (b) Check singularity and solve
det_A = np.linalg.det(A)
rank_A = np.linalg.matrix_rank(A)
print(f"\n(b) det(A) = {det_A:.4f},  rank = {rank_A}  → non-singular: {rank_A == 3}")

T_sol = np.linalg.solve(A, b)
print(f"\n    T1 = {T_sol[0]:.2f} °C")
print(f"    T2 = {T_sol[1]:.2f} °C")
print(f"    T3 = {T_sol[2]:.2f} °C")

# (c) Verify
print(f"\n(c) A @ T_sol = {np.round(A @ T_sol, 6)}")
print(f"    b         = {b}")

# (d) Physical interpretation
# 1. T1 is the highest because it is the only tank directly heated by the jacket
#    (T_J = 120°C). The jacket supplies extra heat that raises T1 well above the
#    cold feed temperature of 25°C.
#
# 2. T3 is the coldest. Heat flows from Tank 1 → Tank 2 → Tank 3 down the
#    temperature gradient. By the time thermal energy reaches Tank 3, it has been
#    diluted by the cold incoming feed at each step, so T3 is the lowest.
#
# 3. Raising T_J to 200°C would increase all three temperatures. T1 would rise the
#    most (directly heated), T2 would rise moderately (gains more from T1),
#    and T3 would also rise but least (furthest from the heat source).

# Optional: verify by changing T_J
T_J_new = 200.0
# Tank 1 RHS: -(mCp_1)*T1in - UA_J*T_J = -(10*25) - 3*200 = -850
b_new = np.array([-(10*25 + 3*T_J_new), -200.0, -150.0])
T_new = np.linalg.solve(A, b_new)
print(f"\n(d) With T_J = {T_J_new}°C:")
print(f"    T1 = {T_new[0]:.2f} °C  (was {T_sol[0]:.2f})")
print(f"    T2 = {T_new[1]:.2f} °C  (was {T_sol[1]:.2f})")
print(f"    T3 = {T_new[2]:.2f} °C  (was {T_sol[2]:.2f})")
